# 12. Same-session Llama wrapper control and energy interaction

New T4 measurements for Paper 744, Reviewer #1, comment 3(b). The earlier C1 and C1-W comparison used separate sessions and prompts. Here one resident AWQ checkpoint runs all five conditions on the same frozen 4,096-token prompt and 64 teacher-forced decode tokens.

| Condition | Attention | Eviction | Reference KV quantization |
| --- | --- | --- | --- |
| C1 | Stock AWQ | Off | Off |
| C1-W | Wrapped AWQ | Off | Off |
| C2 | Wrapped AWQ | On | Off |
| C3-Ref | Wrapped AWQ | Off | On |
| C4-Ref | Wrapped AWQ | On | On |

Run the numbered cells in order in a fresh Google Colab T4 runtime. GPU board power is sampled at 10 Hz. Ten counterbalanced blocks each contain every condition. Warm-ups and the parity probe are outside the analysis. One block is the unit of replication. Conclusions are conditional on this workload and session.

The primary comparison is C1-W minus C1. Two paired interaction estimates use the stock and wrapped baselines, respectively. This is an energy control on the study's custom implementation. It makes no comparison to the authors' PyramidKV benchmark accuracy.

The prompt and the model/cache functions are taken from the accepted Notebook 05. Evidence from completed experiments is read-only. New files go to a unique directory under /content, and the last cell produces a downloadable ZIP.

## 1. Check the T4 and install the study packages

Run in a fresh runtime before importing the model libraries. For a closer match to the original software stack, select the 2026.07 Colab runtime version if available. Dependency installation may take several minutes.

In [1]:
import subprocess

gpu_info = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
).strip().splitlines()
assert len(gpu_info) == 1 and "T4" in gpu_info[0], f"Expected one NVIDIA T4, found {gpu_info}"
%pip install -q "autoawq==0.2.9" "gptqmodel==7.1.0" "turboquant==0.2.0"
%pip install -q "transformers==5.12.1" "numpy>=2" accelerate nvidia-ml-py scipy

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently tak

## 2. Verify the runtime and freeze the workload

The embedded token IDs come from scenario d50-s00 in the accepted Llama prompt manifest. A hash check confirms that the prompt and decode sequence match the workload used for the earlier C1-W control. The session manifest records the pinned checkpoint, software versions, GPU settings, and run order.

In [2]:
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import gc
import hashlib
import importlib.metadata
import json
import math
import random
import threading
import time
from collections.abc import Mapping
from pathlib import Path

import numpy as np
import pynvml
import scipy.stats as stats
import torch
import torch.nn.functional as F
import transformers
import turboquant
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache, set_seed
from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS
from transformers.models.llama.modeling_llama import (
    LlamaAttention,
    apply_rotary_pos_emb,
    eager_attention_forward,
    repeat_kv,
)
from turboquant import TurboQuantCache


from collections import Counter
from datetime import datetime, timezone
import csv
import uuid

# TurboQuant 0.2.0 uses the legacy name for the same trapezoidal rule.
if not hasattr(np, "trapz"):
    np.trapz = np.trapezoid

SCHEMA_VERSION = "track-a-llama-matched-wrapper-interaction-v1"
SEED = 407
MODEL_AWQ = "casperhansen/llama-3.2-3b-instruct-awq"
MODEL_AWQ_REVISION = "272b3bde867b606760447deb9a4d2719fbdfd3ae"
EXPECTED_TQ_SOURCE_SHA256 = "7358e2778c8de6974b0c7212ae8e3dbe00552a96b2e2df0aa42878f239a4c72a"
EXPECTED_PROMPT_SHA256 = "ebdb5cf10c1441038bb5042079dac710dbef8619c9fa35cf0c6bb4d496e3850f"
EXPECTED_TRACE_IDS_SHA256 = "8563b0a2fce11c3ee3091f3f5eafaf50ae25690a8fab0e45c3d3465575833133"
CONTEXT_TOKENS = 4096
FIXED_DECODE_TOKENS = 64
MAX_NEW_TOKENS = 8
MEASURED_BLOCKS = 10
DEFAULT_AVERAGE_BUDGET = 1024
DEFAULT_TQ_BITS = 4
NUM_LAYERS = 28
NUM_Q_HEADS = 24
NUM_KV_HEADS = 8
HEAD_DIM = 128
WINDOW = 32
POOL_KERNEL = 5
PYRAMID_BETA = 20
PARITY_ATOL = PARITY_RTOL = 5e-3
VARIANTS = ("C1", "C1-W", "C2", "C3-Ref", "C4-Ref")
SAMPLE_INTERVAL_S = 0.1

set_seed(SEED)
assert torch.cuda.is_available() and torch.cuda.device_count() == 1
GPU = torch.cuda.get_device_properties(0)
assert "T4" in GPU.name
assert transformers.__version__ == "5.12.1"
for package, version in (("turboquant", "0.2.0"), ("gptqmodel", "7.1.0")):
    assert importlib.metadata.version(package) == version, package
tq_root = Path(turboquant.__file__).parent
tq_source_digest = hashlib.sha256(
    b"".join(path.read_bytes() for path in sorted(tq_root.glob("*.py")))
).hexdigest()
assert tq_source_digest == EXPECTED_TQ_SOURCE_SHA256

pynvml.nvmlInit()
assert pynvml.nvmlDeviceGetCount() == 1
NVML_HANDLE = pynvml.nvmlDeviceGetHandleByIndex(0)

def nvml_text(value):
    return value.decode("utf-8") if isinstance(value, bytes) else str(value)


def gpu_identity():
    return {
        "uuid": nvml_text(pynvml.nvmlDeviceGetUUID(NVML_HANDLE)),
        "name": nvml_text(pynvml.nvmlDeviceGetName(NVML_HANDLE)),
        "power_limit_w": pynvml.nvmlDeviceGetEnforcedPowerLimit(NVML_HANDLE) / 1000.0,
    }


GPU_IDENTITY = gpu_identity()
assert "T4" in GPU_IDENTITY["name"] and GPU_IDENTITY["power_limit_w"] > 0
SESSION_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def canonical_hash(value):
    value = json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def token_ids_hash(ids):
    return hashlib.sha256(",".join(map(str, ids)).encode("ascii")).hexdigest()


def json_ready(value):
    if isinstance(value, dict):
        return {name: json_ready(item) for name, item in value.items()}
    if isinstance(value, (tuple, list)):
        return [json_ready(item) for item in value]
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def atomic_write_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(json_ready(value), indent=2, allow_nan=False) + "\n")
    temporary.replace(path)


def make_schedule(variants, seed):
    count = len(variants)
    assert count == 5
    rng = random.Random(seed)
    labels = list(variants)
    rng.shuffle(labels)
    base = [0, 1, 4, 2, 3]
    orders = [[labels[(index + offset) % count] for index in base] for offset in range(count)]
    orders += [list(reversed(order)) for order in orders]
    rng.shuffle(orders)
    for position in range(count):
        assert Counter(order[position] for order in orders) == {label: 2 for label in variants}
    transitions = Counter(pair for order in orders for pair in zip(order, order[1:]))
    assert len(transitions) == count * (count - 1)
    assert set(transitions.values()) == {2}
    return orders


SCHEDULE = make_schedule(VARIANTS, SEED)
assert len(SCHEDULE) == MEASURED_BLOCKS

W1122 11:22:02.600000 3101 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1122 11:22:02.727000 3101 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [3]:
PROMPT_IDS = [
    128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 2839,
    5033, 220, 2366, 21, 271, 10464, 1193, 279, 2246, 13, 3494, 7041, 832, 15449, 2082, 323, 912, 16540, 13, 128009,
    128006, 882, 128007, 271, 27, 32564, 397, 47502, 220, 16, 35434, 45166, 60199, 13, 578, 3318, 1912, 14242, 27924, 16662,
    1475, 69729, 19179, 13, 19690, 7309, 220, 1049, 15, 574, 22690, 555, 220, 17, 31499, 13, 2360, 3254, 19179, 574,
    12020, 439, 14343, 6029, 369, 24047, 382, 47502, 220, 17, 35434, 31212, 30158, 13, 578, 3318, 1912, 7576, 1855, 5597,
    1603, 279, 1828, 3477, 11008, 13, 19690, 7309, 220, 1049, 16, 574, 22690, 555, 220, 18, 31499, 13, 578, 3477,
    1766, 430, 36899, 23851, 11497, 1455, 315, 279, 2349, 382, 47502, 220, 18, 35434, 5950, 44463, 13, 578, 3318, 1912,
    43565, 3293, 22323, 449, 264, 20700, 26954, 13, 19690, 198, 58, 6182, 3335, 60, 744, 35917, 12279, 9228, 1372, 5829,
    15449, 2082, 469, 16723, 16222, 1544, 627, 7309, 220, 1049, 17, 574, 22690, 555, 220, 19, 31499, 13, 65520, 35363,
    279, 15692, 12447, 369, 279, 1828, 16865, 382, 47502, 220, 19, 35434, 18624, 53080, 13, 578, 3318, 1912, 13912, 7257,
    24654, 2561, 369, 3010, 93596, 13, 19690, 7309, 220, 1049, 18, 574, 22690, 555, 220, 20, 31499, 13, 11359, 5352,
    990, 10968, 389, 8060, 13654, 4856, 1109, 32263, 64862, 382, 47502, 220, 20, 35434, 29487, 38952, 13, 578, 3318, 1912,
    44694, 10741, 1306, 904, 3769, 6683, 2349, 13, 19690, 7309, 220, 1049, 19, 574, 22690, 555, 220, 21, 31499, 13,
    578, 1620, 28782, 2631, 2500, 9678, 19179, 4261, 198, 58, 6182, 3335, 60, 744, 35917, 12, 7842, 68623, 5829, 15449,
    2082, 473, 22908, 29342, 2970, 627, 382, 47502, 220, 21, 35434, 11863, 13709, 13, 578, 3318, 1912, 45777, 4861, 15528,
    5784, 927, 2875, 62954, 16557, 5178, 13, 19690, 7309, 220, 1049, 20, 574, 22690, 555, 220, 22, 31499, 13, 362,
    3010, 25261, 11007, 279, 22106, 719, 32828, 279, 27924, 16430, 382, 47502, 220, 22, 35434, 14830, 43257, 13, 578, 3318,
    1912, 62849, 14348, 13709, 505, 25363, 39455, 13, 19690, 7309, 220, 1049, 21, 574, 22690, 555, 220, 23, 31499, 13,
    578, 1934, 46728, 53823, 7697, 3225, 323, 9539, 19180, 14592, 2819, 382, 47502, 220, 23, 35434, 5044, 24691, 13, 578,
    3318, 1912, 5829, 9678, 12621, 311, 11388, 40240, 198, 58, 6182, 3335, 60, 744, 35917, 12, 5358, 61589, 5829, 15449,
    2082, 469, 25665, 8288, 6028, 627, 24654, 13, 19690, 7309, 220, 1049, 22, 574, 22690, 555, 220, 17, 31499, 13,
    578, 2128, 27470, 2225, 6992, 19622, 323, 28093, 8500, 24654, 382, 47502, 220, 24, 35434, 35335, 16967, 13, 578, 3318,
    1912, 5829, 9678, 12621, 311, 11388, 40240, 24654, 13, 19690, 7309, 220, 1049, 23, 574, 22690, 555, 220, 18, 31499,
    13, 578, 1620, 28782, 2631, 2500, 9678, 19179, 4261, 382, 47502, 220, 605, 35434, 24088, 32758, 13, 578, 3318, 1912,
    43565, 3293, 22323, 449, 264, 20700, 26954, 13, 19690, 7309, 220, 1049, 24, 574, 22690, 555, 220, 19, 31499, 13,
    362, 3010, 25261, 11007, 279, 22106, 719, 32828, 198, 58, 6182, 3335, 60, 744, 35917, 9483, 5726, 2871, 5484, 5829,
    15449, 2082, 426, 18163, 29342, 1399, 627, 279, 27924, 16430, 382, 47502, 220, 806, 35434, 4009, 62137, 13, 578, 3318,
    1912, 13912, 7257, 24654, 2561, 369, 3010, 93596, 13, 19690, 7309, 220, 679, 15, 574, 22690, 555, 220, 20, 31499,
    13, 578, 2128, 27470, 2225, 6992, 19622, 323, 28093, 8500, 24654, 382, 47502, 220, 717, 35434, 3090, 6514, 13, 578,
    3318, 1912, 62849, 14348, 13709, 505, 25363, 39455, 13, 19690, 7309, 220, 679, 16, 574, 22690, 555, 220, 21, 31499,
    13, 11359, 5352, 990, 10968, 389, 8060, 13654, 4856, 1109, 32263, 64862, 382, 47502, 220, 1032, 35434, 9629, 79482, 13,
    578, 3318, 1912, 14242, 27924, 16662, 1475, 69729, 198, 58, 6182, 3335, 60, 744, 35917, 12, 4782, 25939, 5829, 15449,
    2082, 362, 15741, 29342, 2790, 627, 19179, 13, 19690, 7309, 220, 679, 17, 574, 22690, 555, 220, 22, 31499, 13,
    2360, 3254, 19179, 574, 12020, 439, 14343, 6029, 369, 24047, 382, 47502, 220, 975, 35434, 12585, 38711, 13, 578, 3318,
    1912, 7576, 1855, 5597, 1603, 279, 1828, 3477, 11008, 13, 19690, 7309, 220, 679, 18, 574, 22690, 555, 220, 23,
    31499, 13, 578, 3477, 1766, 430, 36899, 23851, 11497, 1455, 315, 279, 2349, 382, 47502, 220, 868, 35434, 19641, 6492,
    13, 578, 3318, 1912, 45777, 4861, 15528, 5784, 927, 2875, 62954, 16557, 5178, 13, 19690, 7309, 220, 679, 19, 574,
    22690, 555, 220, 17, 31499, 13, 65520, 198, 58, 6182, 3335, 60, 744, 35917, 8288, 2891, 13974, 5829, 15449, 2082,
    435, 23812, 8288, 605, 627, 35363, 279, 15692, 12447, 369, 279, 1828, 16865, 382, 47502, 220, 845, 35434, 19979, 7649,
    13, 578, 3318, 1912, 44694, 10741, 1306, 904, 3769, 6683, 2349, 13, 19690, 7309, 220, 679, 20, 574, 22690, 555,
    220, 18, 31499, 13, 578, 1934, 46728, 53823, 7697, 3225, 323, 9539, 19180, 14592, 2819, 382, 47502, 220, 1114, 35434,
    31212, 30158, 13, 578, 3318, 1912, 45777, 4861, 15528, 5784, 927, 2875, 62954, 16557, 5178, 13, 19690, 7309, 220, 679,
    21, 574, 22690, 555, 220, 19, 31499, 13, 362, 3010, 25261, 11007, 279, 22106, 719, 32828, 279, 27924, 16430, 382,
    47502, 220, 972, 35434, 18624, 53080, 13, 198, 58, 6182, 3335, 60, 744, 35917, 12, 79175, 5829, 15449, 2082, 469,
    25358, 9483, 1272, 627, 578, 3318, 1912, 5829, 9678, 12621, 311, 11388, 40240, 24654, 13, 19690, 7309, 220, 679, 22,
    574, 22690, 555, 220, 20, 31499, 13, 65520, 35363, 279, 15692, 12447, 369, 279, 1828, 16865, 382, 47502, 220, 777,
    35434, 4009, 62137, 13, 578, 3318, 1912, 44694, 10741, 1306, 904, 3769, 6683, 2349, 13, 19690, 7309, 220, 679, 23,
    574, 22690, 555, 220, 21, 31499, 13, 11359, 5352, 990, 10968, 389, 8060, 13654, 4856, 1109, 32263, 64862, 382, 47502,
    220, 508, 35434, 12585, 38711, 13, 578, 3318, 1912, 7576, 1855, 5597, 1603, 279, 1828, 3477, 11008, 13, 19690, 7309,
    220, 679, 24, 574, 22690, 198, 58, 6182, 3335, 60, 744, 35917, 12, 4031, 33, 59169, 5829, 15449, 2082, 480,
    21618, 11500, 2618, 627, 555, 220, 22, 31499, 13, 2360, 3254, 19179, 574, 12020, 439, 14343, 6029, 369, 24047, 382,
    47502, 220, 1691, 35434, 35335, 16967, 13, 578, 3318, 1912, 43565, 3293, 22323, 449, 264, 20700, 26954, 13, 19690, 7309,
    220, 2366, 15, 574, 22690, 555, 220, 23, 31499, 13, 578, 1620, 28782, 2631, 2500, 9678, 19179, 4261, 382, 47502,
    220, 1313, 35434, 3090, 6514, 13, 578, 3318, 1912, 14242, 27924, 16662, 1475, 69729, 19179, 13, 19690, 7309, 220, 2366,
    16, 574, 22690, 555, 220, 17, 31499, 13, 578, 1934, 46728, 53823, 7697, 3225, 323, 9539, 19180, 14592, 2819, 382,
    47502, 220, 1419, 35434, 29487, 38952, 198, 58, 6182, 3335, 60, 744, 35917, 6830, 1539, 52577, 5829, 15449, 2082, 356,
    7529, 5364, 6083, 627, 13, 578, 3318, 1912, 62849, 14348, 13709, 505, 25363, 39455, 13, 19690, 7309, 220, 2366, 17,
    574, 22690, 555, 220, 18, 31499, 13, 578, 2128, 27470, 2225, 6992, 19622, 323, 28093, 8500, 24654, 382, 47502, 220,
    1187, 35434, 19641, 6492, 13, 578, 3318, 1912, 13912, 7257, 24654, 2561, 369, 3010, 93596, 13, 19690, 7309, 220, 2366,
    18, 574, 22690, 555, 220, 19, 31499, 13, 578, 3477, 1766, 430, 36899, 23851, 11497, 1455, 315, 279, 2349, 382,
    47502, 220, 914, 35434, 5044, 24691, 13, 578, 3318, 1912, 45777, 4861, 15528, 5784, 927, 2875, 62954, 16557, 5178, 13,
    19690, 7309, 220, 2366, 19, 198, 58, 6182, 3335, 60, 744, 35917, 12, 72667, 5726, 5829, 15449, 2082, 356, 19808,
    29342, 914, 627, 574, 22690, 555, 220, 20, 31499, 13, 578, 3477, 1766, 430, 36899, 23851, 11497, 1455, 315, 279,
    2349, 382, 47502, 220, 1627, 35434, 14830, 43257, 13, 578, 3318, 1912, 7576, 1855, 5597, 1603, 279, 1828, 3477, 11008,
    13, 19690, 7309, 220, 2366, 20, 574, 22690, 555, 220, 21, 31499, 13, 578, 1934, 46728, 53823, 7697, 3225, 323,
    9539, 19180, 14592, 2819, 382, 47502, 220, 1544, 35434, 11863, 13709, 13, 578, 3318, 1912, 62849, 14348, 13709, 505, 25363,
    39455, 13, 19690, 7309, 220, 2366, 21, 574, 22690, 555, 220, 22, 31499, 13, 578, 1620, 28782, 2631, 2500, 9678,
    19179, 4261, 382, 47502, 220, 198, 58, 6182, 3335, 60, 744, 35917, 31650, 965, 715, 5829, 15449, 2082, 480, 17451,
    8288, 5495, 627, 1591, 35434, 9629, 79482, 13, 578, 3318, 1912, 14242, 27924, 16662, 1475, 69729, 19179, 13, 19690, 7309,
    220, 2366, 22, 574, 22690, 555, 220, 23, 31499, 13, 11359, 5352, 990, 10968, 389, 8060, 13654, 4856, 1109, 32263,
    64862, 382, 47502, 220, 1682, 35434, 24088, 32758, 13, 578, 3318, 1912, 5829, 9678, 12621, 311, 11388, 40240, 24654, 13,
    19690, 7309, 220, 2366, 23, 574, 22690, 555, 220, 17, 31499, 13, 2360, 3254, 19179, 574, 12020, 439, 14343, 6029,
    369, 24047, 382, 47502, 220, 966, 35434, 5950, 44463, 13, 578, 3318, 1912, 13912, 7257, 24654, 2561, 369, 3010, 93596,
    13, 19690, 7309, 220, 2366, 198, 58, 6182, 3335, 60, 744, 35917, 12, 5876, 3065, 57, 5829, 15449, 2082, 362,
    16085, 11500, 5313, 627, 24, 574, 22690, 555, 220, 18, 31499, 13, 65520, 35363, 279, 15692, 12447, 369, 279, 1828,
    16865, 382, 47502, 220, 2148, 35434, 45166, 60199, 13, 578, 3318, 1912, 44694, 10741, 1306, 904, 3769, 6683, 2349, 13,
    19690, 7309, 220, 9639, 15, 574, 22690, 555, 220, 19, 31499, 13, 362, 3010, 25261, 11007, 279, 22106, 719, 32828,
    279, 27924, 16430, 382, 47502, 220, 843, 35434, 19979, 7649, 13, 578, 3318, 1912, 43565, 3293, 22323, 449, 264, 20700,
    26954, 13, 19690, 7309, 220, 9639, 16, 574, 22690, 555, 220, 20, 31499, 13, 578, 2128, 27470, 2225, 6992, 19622,
    323, 28093, 8500, 24654, 382, 198, 58, 6182, 3335, 60, 744, 35917, 30237, 946, 15445, 5829, 15449, 2082, 473, 22905,
    5364, 6086, 627, 47502, 220, 1644, 35434, 18624, 53080, 13, 578, 3318, 1912, 7576, 1855, 5597, 1603, 279, 1828, 3477,
    11008, 13, 19690, 7309, 220, 9639, 17, 574, 22690, 555, 220, 21, 31499, 13, 2360, 3254, 19179, 574, 12020, 439,
    14343, 6029, 369, 24047, 382, 47502, 220, 1958, 35434, 3090, 6514, 13, 578, 3318, 1912, 62849, 14348, 13709, 505, 25363,
    39455, 13, 19690, 7309, 220, 9639, 18, 574, 22690, 555, 220, 22, 31499, 13, 578, 3477, 1766, 430, 36899, 23851,
    11497, 1455, 315, 279, 2349, 382, 47502, 220, 1758, 35434, 31212, 30158, 13, 578, 3318, 1912, 5829, 9678, 12621, 311,
    11388, 40240, 24654, 13, 198, 58, 6182, 3335, 60, 744, 35917, 13299, 1750, 9628, 5829, 15449, 2082, 435, 7743, 9483,
    966, 627, 19690, 7309, 220, 9639, 19, 574, 22690, 555, 220, 23, 31499, 13, 578, 1934, 46728, 53823, 7697, 3225,
    323, 9539, 19180, 14592, 2819, 382, 47502, 220, 1927, 35434, 9629, 79482, 13, 578, 3318, 1912, 44694, 10741, 1306, 904,
    3769, 6683, 2349, 13, 19690, 7309, 220, 9639, 20, 574, 22690, 555, 220, 17, 31499, 13, 362, 3010, 25261, 11007,
    279, 22106, 719, 32828, 279, 27924, 16430, 382, 47502, 220, 1806, 35434, 14830, 43257, 13, 578, 3318, 1912, 14242, 27924,
    16662, 1475, 69729, 19179, 13, 19690, 7309, 220, 9639, 21, 574, 22690, 555, 220, 18, 31499, 13, 65520, 35363, 279,
    15692, 12447, 369, 279, 198, 58, 6182, 3335, 60, 744, 35917, 7813, 1507, 946, 5829, 15449, 2082, 426, 23717, 29342,
    2946, 627, 1828, 16865, 382, 47502, 220, 1987, 35434, 12585, 38711, 13, 578, 3318, 1912, 43565, 3293, 22323, 449, 264,
    20700, 26954, 13, 19690, 7309, 220, 9639, 22, 574, 22690, 555, 220, 19, 31499, 13, 11359, 5352, 990, 10968, 389,
    8060, 13654, 4856, 1109, 32263, 64862, 382, 47502, 220, 2137, 35434, 19979, 7649, 13, 578, 3318, 1912, 45777, 4861, 15528,
    5784, 927, 2875, 62954, 16557, 5178, 13, 19690, 7309, 220, 9639, 23, 574, 22690, 555, 220, 20, 31499, 13, 578,
    1620, 28782, 2631, 2500, 9678, 19179, 4261, 382, 47502, 220, 1272, 35434, 29487, 38952, 13, 578, 3318, 1912, 13912, 7257,
    24654, 2561, 369, 198, 58, 6182, 3335, 60, 744, 35917, 5364, 946, 33004, 5829, 15449, 2082, 480, 21741, 11500, 2148,
    627, 3010, 93596, 13, 19690, 7309, 220, 9639, 24, 574, 22690, 555, 220, 21, 31499, 13, 578, 2128, 27470, 2225,
    6992, 19622, 323, 28093, 8500, 24654, 382, 47502, 220, 3174, 35434, 5044, 24691, 13, 578, 3318, 1912, 62849, 14348, 13709,
    505, 25363, 39455, 13, 19690, 7309, 220, 7854, 15, 574, 22690, 555, 220, 22, 31499, 13, 578, 3477, 1766, 430,
    36899, 23851, 11497, 1455, 315, 279, 2349, 382, 47502, 220, 2983, 35434, 4009, 62137, 13, 578, 3318, 1912, 5829, 9678,
    12621, 311, 11388, 40240, 24654, 13, 19690, 7309, 220, 7854, 16, 574, 22690, 555, 220, 23, 31499, 13, 578, 1620,
    28782, 2631, 2500, 198, 58, 6182, 3335, 60, 744, 35917, 12, 44787, 5829, 15449, 2082, 426, 23480, 16222, 4643, 627,
    47502, 220, 16, 35434, 14830, 43257, 13, 578, 3318, 1912, 62849, 14348, 13709, 505, 25363, 39455, 13, 19690, 7309, 220,
    1049, 15, 574, 22690, 555, 220, 17, 31499, 13, 65520, 35363, 279, 15692, 12447, 369, 279, 1828, 16865, 382, 47502,
    220, 17, 35434, 11863, 13709, 13, 578, 3318, 1912, 14242, 27924, 16662, 1475, 69729, 19179, 13, 19690, 7309, 220, 1049,
    16, 574, 22690, 555, 220, 18, 31499, 13, 578, 1620, 28782, 2631, 2500, 9678, 19179, 4261, 382, 47502, 220, 18,
    35434, 3090, 6514, 13, 578, 3318, 1912, 13912, 7257, 24654, 2561, 369, 3010, 93596, 13, 19690, 7309, 220, 1049, 17,
    574, 22690, 555, 220, 19, 31499, 13, 198, 58, 6182, 3335, 60, 744, 35917, 12, 878, 21587, 984, 5829, 15449,
    2082, 480, 16104, 29342, 4044, 627, 578, 3477, 1766, 430, 36899, 23851, 11497, 1455, 315, 279, 2349, 382, 47502, 220,
    19, 35434, 18624, 53080, 13, 578, 3318, 1912, 45777, 4861, 15528, 5784, 927, 2875, 62954, 16557, 5178, 13, 19690, 7309,
    220, 1049, 18, 574, 22690, 555, 220, 20, 31499, 13, 578, 2128, 27470, 2225, 6992, 19622, 323, 28093, 8500, 24654,
    382, 47502, 220, 20, 35434, 4009, 62137, 13, 578, 3318, 1912, 43565, 3293, 22323, 449, 264, 20700, 26954, 13, 19690,
    7309, 220, 1049, 19, 574, 22690, 555, 220, 21, 31499, 13, 578, 1934, 46728, 53823, 7697, 3225, 323, 9539, 19180,
    14592, 2819, 382, 47502, 220, 21, 35434, 12585, 38711, 13, 578, 3318, 1912, 7576, 198, 58, 6182, 3335, 60, 744,
    35917, 12, 3472, 790, 50151, 5829, 15449, 2082, 362, 21038, 5364, 4218, 627, 1855, 5597, 1603, 279, 1828, 3477, 11008,
    13, 19690, 7309, 220, 1049, 20, 574, 22690, 555, 220, 22, 31499, 13, 11359, 5352, 990, 10968, 389, 8060, 13654,
    4856, 1109, 32263, 64862, 382, 47502, 220, 22, 35434, 45166, 60199, 13, 578, 3318, 1912, 5829, 9678, 12621, 311, 11388,
    40240, 24654, 13, 19690, 7309, 220, 1049, 21, 574, 22690, 555, 220, 23, 31499, 13, 362, 3010, 25261, 11007, 279,
    22106, 719, 32828, 279, 27924, 16430, 382, 47502, 220, 23, 35434, 5044, 24691, 13, 578, 3318, 1912, 44694, 10741, 1306,
    904, 3769, 6683, 2349, 13, 19690, 7309, 220, 1049, 22, 574, 22690, 555, 220, 17, 31499, 13, 2360, 3254, 19179,
    198, 58, 6182, 3335, 60, 744, 35917, 7424, 984, 34, 5338, 5829, 15449, 2082, 473, 7994, 16222, 2096, 627, 574,
    12020, 439, 14343, 6029, 369, 24047, 382, 47502, 220, 24, 35434, 9629, 79482, 13, 578, 3318, 1912, 62849, 14348, 13709,
    505, 25363, 39455, 13, 19690, 7309, 220, 1049, 23, 574, 22690, 555, 220, 18, 31499, 13, 2360, 3254, 19179, 574,
    12020, 439, 14343, 6029, 369, 24047, 382, 47502, 220, 605, 35434, 29487, 38952, 13, 578, 3318, 1912, 7576, 1855, 5597,
    1603, 279, 1828, 3477, 11008, 13, 19690, 7309, 220, 1049, 24, 574, 22690, 555, 220, 19, 31499, 13, 11359, 5352,
    990, 10968, 389, 8060, 13654, 4856, 1109, 32263, 64862, 382, 47502, 220, 806, 35434, 19641, 6492, 13, 578, 3318, 1912,
    5829, 9678, 12621, 311, 11388, 40240, 24654, 198, 58, 6182, 3335, 60, 744, 35917, 7424, 984, 5910, 5829, 15449, 2082,
    356, 12112, 29342, 4161, 627, 13, 19690, 7309, 220, 679, 15, 574, 22690, 555, 220, 20, 31499, 13, 578, 3477,
    1766, 430, 36899, 23851, 11497, 1455, 315, 279, 2349, 382, 47502, 220, 717, 35434, 31212, 30158, 13, 578, 3318, 1912,
    44694, 10741, 1306, 904, 3769, 6683, 2349, 13, 19690, 7309, 220, 679, 16, 574, 22690, 555, 220, 21, 31499, 13,
    578, 1620, 28782, 2631, 2500, 9678, 19179, 4261, 382, 47502, 220, 1032, 35434, 5950, 44463, 13, 578, 3318, 1912, 13912,
    7257, 24654, 2561, 369, 3010, 93596, 13, 19690, 7309, 220, 679, 17, 574, 22690, 555, 220, 22, 31499, 13, 362,
    3010, 25261, 11007, 279, 22106, 719, 32828, 279, 27924, 16430, 382, 47502, 220, 198, 58, 6182, 3335, 60, 744, 35917,
    5364, 946, 691, 643, 5829, 15449, 2082, 426, 10866, 5364, 4767, 627, 975, 35434, 35335, 16967, 13, 578, 3318, 1912,
    43565, 3293, 22323, 449, 264, 20700, 26954, 13, 19690, 7309, 220, 679, 18, 574, 22690, 555, 220, 23, 31499, 13,
    578, 2128, 27470, 2225, 6992, 19622, 323, 28093, 8500, 24654, 382, 47502, 220, 868, 35434, 24088, 32758, 13, 578, 3318,
    1912, 45777, 4861, 15528, 5784, 927, 2875, 62954, 16557, 5178, 13, 19690, 7309, 220, 679, 19, 574, 22690, 555, 220,
    17, 31499, 13, 65520, 35363, 279, 15692, 12447, 369, 279, 1828, 16865, 382, 47502, 220, 845, 35434, 19979, 7649, 13,
    578, 3318, 1912, 14242, 27924, 16662, 1475, 69729, 19179, 13, 19690, 7309, 220, 679, 20, 574, 22690, 555, 220, 18,
    198, 58, 6182, 3335, 60, 744, 35917, 12, 878, 21587, 5829, 15449, 2082, 480, 15741, 16222, 2495, 627, 31499, 13,
    578, 1934, 46728, 53823, 7697, 3225, 323, 9539, 19180, 14592, 2819, 382, 47502, 220, 1114, 35434, 14830, 43257, 13, 578,
    3318, 1912, 5829, 9678, 12621, 311, 11388, 40240, 24654, 13, 19690, 7309, 220, 679, 21, 574, 22690, 555, 220, 19,
    31499, 13, 2360, 3254, 19179, 574, 12020, 439, 14343, 6029, 369, 24047, 382, 47502, 220, 972, 35434, 4009, 62137, 13,
    578, 3318, 1912, 14242, 27924, 16662, 1475, 69729, 19179, 13, 19690, 7309, 220, 679, 22, 574, 22690, 555, 220, 20,
    31499, 13, 578, 3477, 1766, 430, 36899, 23851, 11497, 1455, 315, 279, 2349, 382, 47502, 220, 777, 35434, 19979, 7649,
    13, 578, 3318, 1912, 45777, 4861, 198, 58, 6182, 3335, 60, 744, 35917, 12, 21695, 1777, 790, 5829, 15449, 2082,
    435, 26437, 16222, 2096, 627, 15528, 5784, 927, 2875, 62954, 16557, 5178, 13, 19690, 7309, 220, 679, 23, 574, 22690,
    555, 220, 21, 31499, 13, 578, 2128, 27470, 2225, 6992, 19622, 323, 28093, 8500, 24654, 382, 47502, 220, 508, 35434,
    11863, 13709, 13, 578, 3318, 1912, 44694, 10741, 1306, 904, 3769, 6683, 2349, 13, 19690, 7309, 220, 679, 24, 574,
    22690, 555, 220, 22, 31499, 13, 65520, 35363, 279, 15692, 12447, 369, 279, 1828, 16865, 382, 47502, 220, 1691, 35434,
    45166, 60199, 13, 578, 3318, 1912, 13912, 7257, 24654, 2561, 369, 3010, 93596, 13, 19690, 7309, 220, 2366, 15, 574,
    22690, 555, 220, 23, 31499, 13, 578, 1620, 28782, 2631, 2500, 9678, 198, 58, 6182, 3335, 60, 744, 35917, 6354,
    2864, 18836, 5829, 15449, 2082, 435, 23654, 29342, 1627, 627, 19179, 4261, 382, 47502, 220, 1313, 35434, 35335, 16967, 13,
    578, 3318, 1912, 62849, 14348, 13709, 505, 25363, 39455, 13, 19690, 7309, 220, 2366, 16, 574, 22690, 555, 220, 17,
    31499, 13, 11359, 5352, 990, 10968, 389, 8060, 13654, 4856, 1109, 32263, 64862, 382, 47502, 220, 1419, 35434, 5950, 44463,
    13, 578, 3318, 1912, 43565, 3293, 22323, 449, 264, 20700, 26954, 13, 19690, 7309, 220, 2366, 17, 574, 22690, 555,
    220, 18, 31499, 13, 362, 3010, 25261, 11007, 279, 22106, 719, 32828, 279, 27924, 16430, 382, 47502, 220, 1187, 35434,
    12585, 38711, 13, 578, 3318, 1912, 7576, 1855, 5597, 1603, 279, 1828, 3477, 11008, 13, 19690, 7309, 220, 198, 58,
    6182, 3335, 60, 744, 35917, 7813, 1507, 37078, 5829, 15449, 2082, 356, 24427, 5364, 843, 627, 2366, 18, 574, 22690,
    555, 220, 19, 31499, 13, 578, 1934, 46728, 53823, 7697, 3225, 323, 9539, 19180, 14592, 2819, 382, 47502, 220, 914,
    35434, 9629, 79482, 13, 578, 3318, 1912, 5829, 9678, 12621, 311, 11388, 40240, 24654, 13, 19690, 7309, 220, 2366, 19,
    574, 22690, 555, 220, 20, 31499, 13, 11359, 5352, 990, 10968, 389, 8060, 13654, 4856, 1109, 32263, 64862, 382, 47502,
    220, 1627, 35434, 31212, 30158, 13, 578, 3318, 1912, 7576, 1855, 5597, 1603, 279, 1828, 3477, 11008, 13, 19690, 7309,
    220, 2366, 20, 574, 22690, 555, 220, 21, 31499, 13, 578, 1934, 46728, 53823, 7697, 3225, 323, 9539, 19180, 14592,
    2819, 382, 47502, 198, 58, 6182, 3335, 60, 744, 35917, 11529, 50164, 878, 31650, 5829, 15449, 2082, 473, 10841, 29342,
    4161, 627, 220, 1544, 35434, 5044, 24691, 13, 578, 3318, 1912, 62849, 14348, 13709, 505, 25363, 39455, 13, 19690, 7309,
    220, 2366, 21, 574, 22690, 555, 220, 22, 31499, 13, 578, 1620, 28782, 2631, 2500, 9678, 19179, 4261, 382, 47502,
    220, 1591, 35434, 18624, 53080, 13, 578, 3318, 1912, 13912, 7257, 24654, 2561, 369, 3010, 93596, 13, 19690, 7309, 220,
    2366, 22, 574, 22690, 555, 220, 23, 31499, 13, 578, 2128, 27470, 2225, 6992, 19622, 323, 28093, 8500, 24654, 382,
    47502, 220, 1682, 35434, 19641, 6492, 13, 578, 3318, 1912, 14242, 27924, 16662, 1475, 69729, 19179, 13, 19690, 7309, 220,
    2366, 23, 574, 22690, 555, 220, 17, 31499, 13, 198, 58, 6182, 3335, 60, 744, 35917, 12, 5726, 42122, 5829,
    15449, 2082, 356, 14274, 9483, 868, 627, 578, 3477, 1766, 430, 36899, 23851, 11497, 1455, 315, 279, 2349, 382, 47502,
    220, 966, 35434, 24088, 32758, 13, 578, 3318, 1912, 43565, 3293, 22323, 449, 264, 20700, 26954, 13, 19690, 7309, 220,
    2366, 24, 574, 22690, 555, 220, 18, 31499, 13, 362, 3010, 25261, 11007, 279, 22106, 719, 32828, 279, 27924, 16430,
    382, 47502, 220, 2148, 35434, 29487, 38952, 13, 578, 3318, 1912, 44694, 10741, 1306, 904, 3769, 6683, 2349, 13, 19690,
    7309, 220, 9639, 15, 574, 22690, 555, 220, 19, 31499, 13, 2360, 3254, 19179, 574, 12020, 439, 14343, 6029, 369,
    24047, 382, 47502, 220, 843, 35434, 3090, 6514, 13, 578, 3318, 1912, 45777, 4861, 198, 58, 6182, 3335, 60, 744,
    35917, 12279, 9228, 1372, 31650, 5829, 15449, 2082, 480, 10680, 9483, 6069, 627, 15528, 5784, 927, 2875, 62954, 16557, 5178,
    13, 19690, 7309, 220, 9639, 16, 574, 22690, 555, 220, 20, 31499, 13, 65520, 35363, 279, 15692, 12447, 369, 279,
    1828, 16865, 382, 47502, 220, 1644, 35434, 12585, 38711, 13, 578, 3318, 1912, 45777, 4861, 15528, 5784, 927, 2875, 62954,
    16557, 5178, 13, 19690, 7309, 220, 9639, 17, 574, 22690, 555, 220, 21, 31499, 13, 2360, 3254, 19179, 574, 12020,
    439, 14343, 6029, 369, 24047, 382, 47502, 220, 1958, 35434, 14830, 43257, 13, 578, 3318, 1912, 43565, 3293, 22323, 449,
    264, 20700, 26954, 13, 19690, 7309, 220, 9639, 18, 574, 22690, 555, 220, 22, 31499, 13, 578, 1620, 28782, 2631,
    2500, 198, 58, 6182, 3335, 60, 744, 35917, 11529, 50164, 878, 5829, 15449, 2082, 362, 24425, 11500, 1135, 627, 9678,
    19179, 4261, 382, 47502, 220, 1758, 35434, 31212, 30158, 13, 578, 3318, 1912, 44694, 10741, 1306, 904, 3769, 6683, 2349,
    13, 19690, 7309, 220, 9639, 19, 574, 22690, 555, 220, 23, 31499, 13, 578, 1934, 46728, 53823, 7697, 3225, 323,
    9539, 19180, 14592, 2819, 382, 47502, 220, 1927, 35434, 4009, 62137, 13, 578, 3318, 1912, 62849, 14348, 13709, 505, 25363,
    39455, 13, 19690, 7309, 220, 9639, 20, 574, 22690, 555, 220, 17, 31499, 13, 578, 3477, 1766, 430, 36899, 23851,
    11497, 1455, 315, 279, 2349, 382, 47502, 220, 1806, 35434, 18624, 53080, 13, 578, 3318, 1912, 7576, 1855, 5597, 1603,
    279, 1828, 3477, 11008, 13, 19690, 7309, 198, 58, 6182, 3335, 60, 744, 35917, 12278, 1950, 9851, 5829, 15449, 2082,
    423, 24626, 29342, 1927, 627, 220, 9639, 21, 574, 22690, 555, 220, 18, 31499, 13, 11359, 5352, 990, 10968, 389,
    8060, 13654, 4856, 1109, 32263, 64862, 382, 47502, 220, 1987, 35434, 19641, 6492, 13, 578, 3318, 1912, 13912, 7257, 24654,
    2561, 369, 3010, 93596, 13, 19690, 7309, 220, 9639, 22, 574, 22690, 555, 220, 19, 31499, 13, 65520, 35363, 279,
    15692, 12447, 369, 279, 1828, 16865, 382, 47502, 220, 2137, 35434, 11863, 13709, 13, 578, 3318, 1912, 5829, 9678, 12621,
    311, 11388, 40240, 24654, 13, 19690, 7309, 220, 9639, 23, 574, 22690, 555, 220, 20, 31499, 13, 362, 3010, 25261,
    11007, 279, 22106, 719, 32828, 279, 27924, 16430, 382, 47502, 220, 1272, 198, 58, 6182, 3335, 60, 744, 35917, 11419,
    9377, 49952, 49, 5829, 15449, 2082, 426, 6860, 29342, 3226, 627, 35434, 29487, 38952, 13, 578, 3318, 1912, 14242, 27924,
    16662, 1475, 69729, 19179, 13, 19690, 7309, 220, 9639, 24, 574, 22690, 555, 220, 21, 31499, 13, 578, 2128, 27470,
    2225, 6992, 19622, 323, 28093, 8500, 24654, 382, 47502, 220, 3174, 35434, 3090, 6514, 13, 578, 3318, 1912, 45777, 4861,
    15528, 5784, 927, 2875, 62954, 16557, 5178, 13, 19690, 7309, 220, 7854, 15, 574, 22690, 555, 220, 22, 31499, 13,
    578, 1934, 46728, 53823, 7697, 3225, 323, 9539, 19180, 14592, 2819, 382, 47502, 220, 2983, 35434, 9629, 79482, 13, 578,
    3318, 1912, 5829, 9678, 12621, 311, 11388, 40240, 24654, 13, 19690, 7309, 220, 7854, 16, 574, 22690, 555, 198, 524,
    32564, 397, 3923, 374, 279, 2082, 369, 35917, 12, 44787, 30, 128009, 128006, 78191, 128007, 271,
]
assert len(PROMPT_IDS) == CONTEXT_TOKENS
assert token_ids_hash(PROMPT_IDS) == EXPECTED_PROMPT_SHA256
PROMPT = {"input_ids": PROMPT_IDS, "input_ids_sha256": EXPECTED_PROMPT_SHA256}

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_AWQ, revision=MODEL_AWQ_REVISION, trust_remote_code=True,
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

TRACE_TEXT = (
    "A controlled decode trace keeps the generated workload identical across every "
    "compression configuration. This prevents early end tokens from changing the amount "
    "of measured work and supports a fair energy comparison. "
) * 4
TRACE_IDS = tokenizer.encode(TRACE_TEXT, add_special_tokens=False)[:FIXED_DECODE_TOKENS]
assert len(TRACE_IDS) == FIXED_DECODE_TOKENS
TRACE_IDS_SHA256 = token_ids_hash(TRACE_IDS)
TRACE_TEXT_SHA256 = hashlib.sha256(TRACE_TEXT.encode("utf-8")).hexdigest()


assert TRACE_IDS_SHA256 == EXPECTED_TRACE_IDS_SHA256

def require_frozen_gate():
    assert token_ids_hash(PROMPT_IDS) == EXPECTED_PROMPT_SHA256
    assert token_ids_hash(TRACE_IDS) == EXPECTED_TRACE_IDS_SHA256
    assert gpu_identity() == GPU_IDENTITY, "GPU or power limit changed"


PROTOCOL = {
    "schema_version": SCHEMA_VERSION,
    "session_id": SESSION_ID,
    "gpu": GPU_IDENTITY,
    "model": MODEL_AWQ,
    "model_revision": MODEL_AWQ_REVISION,
    "prompt_id": "d50-s00",
    "input_ids_sha256": EXPECTED_PROMPT_SHA256,
    "trace_ids_sha256": EXPECTED_TRACE_IDS_SHA256,
    "context_tokens": CONTEXT_TOKENS,
    "fixed_decode_tokens": FIXED_DECODE_TOKENS,
    "average_eviction_budget": DEFAULT_AVERAGE_BUDGET,
    "observation_window": WINDOW,
    "pool_kernel": POOL_KERNEL,
    "pyramid_beta": PYRAMID_BETA,
    "nominal_tq_bits": DEFAULT_TQ_BITS,
    "power_sample_interval_s": SAMPLE_INTERVAL_S,
    "primary_metric": "raw GPU-board prefill plus decode joules",
    "primary_contrast": "C1-W minus C1",
    "interaction_formula": "(C4-Ref - C3-Ref) - (C2 - baseline)",
    "ci": "two-sided 95% Student t interval on block-level paired contrasts",
    "schedule_seed": SEED,
    "schedule": SCHEDULE,
    "attention_backend": "sdpa",
    "versions": {
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "driver": nvml_text(pynvml.nvmlSystemGetDriverVersion()),
        **{p: importlib.metadata.version(p) for p in ("transformers", "autoawq", "gptqmodel", "turboquant", "numpy", "scipy", "nvidia-ml-py")},
    },
    "turboquant_source_sha256": tq_source_digest,
    "numpy_trapz_alias": np.trapz is np.trapezoid,
}
EXPERIMENT_FINGERPRINT = canonical_hash(PROTOCOL)[:16]
SAVE_DIR = Path("/content") / f"CSE407_llama_matched_wrapper_interaction_v1_{SESSION_ID}"
BLOCK_DIR = SAVE_DIR / "blocks"
POWER_TRACE_DIR = SAVE_DIR / "power_traces"
SAVE_DIR.mkdir(exist_ok=False)
BLOCK_DIR.mkdir()
POWER_TRACE_DIR.mkdir()
atomic_write_json(SAVE_DIR / "run_manifest.json", {**PROTOCOL, "fingerprint": EXPERIMENT_FINGERPRINT})
atomic_write_json(SAVE_DIR / "workload.json", {
    **PROMPT, "trace_ids": TRACE_IDS, "trace_ids_sha256": TRACE_IDS_SHA256,
})
print(f"{GPU.name} | {EXPERIMENT_FINGERPRINT} | {SAVE_DIR}")

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.6k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Tesla T4 | 240d4ec29411a934 | /content/CSE407_llama_matched_wrapper_interaction_v1_20261122T112205Z-348460e2


## 3. Define and verify the attention paths

The attention wrapper, custom cache eviction, and cache-path checks below reuse the accepted Llama implementation. Every run checks cache lengths and active-layer updates. The stock and wrapped baselines also pass a frozen-prompt parity probe before measurement.

In [4]:
def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


def validate_model_architecture(model):
    config = model.config
    assert config.model_type == "llama"
    assert config.num_hidden_layers == NUM_LAYERS
    assert config.num_attention_heads == NUM_Q_HEADS
    assert config.num_key_value_heads == NUM_KV_HEADS
    assert getattr(config, "head_dim", config.hidden_size // NUM_Q_HEADS) == HEAD_DIM
    assert config.vocab_size == len(tokenizer)


def load_awq():
    require_frozen_gate()
    assert LlamaAttention.forward is ORIGINAL_LLAMA_FORWARD
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_AWQ,
        revision=MODEL_AWQ_REVISION,
        device_map={"": 0},
        attn_implementation="sdpa",
        trust_remote_code=True,
    ).eval()
    validate_model_architecture(model)
    raw = model.config.quantization_config
    quant = raw.to_dict() if hasattr(raw, "to_dict") else dict(raw)
    assert quant["quant_method"] == "awq"
    assert quant["bits"] == 4 and quant["group_size"] == 128
    return model


def build_pyramid_budgets(
    layers=NUM_LAYERS,
    average_budget=DEFAULT_AVERAGE_BUDGET,
    window=WINDOW,
    beta=PYRAMID_BETA,
):
    average_budget = int(average_budget)
    assert layers > 1 and average_budget > window and beta > 0
    old_average = average_budget - window
    last = round(old_average / beta)
    first = 2 * old_average - last
    budgets = (np.linspace(first, last, layers).round().astype(int) + window).tolist()
    assert len(budgets) == layers and int(np.mean(budgets)) == average_budget
    assert min(budgets) >= window and all(a >= b for a, b in zip(budgets, budgets[1:]))
    return budgets


PYRAMID_BUDGETS = build_pyramid_budgets()
VARIANT_PATHS = {
    "C1": {"wrapper": False, "eviction": False, "tq": False},
    "C1-W": {"wrapper": True, "eviction": False, "tq": False},
    "C2": {"wrapper": True, "eviction": True, "tq": False},
    "C3-Ref": {"wrapper": True, "eviction": False, "tq": True},
    "C4-Ref": {"wrapper": True, "eviction": True, "tq": True},
}
PKV_LOG = []
CACHE_UPDATE_LOG = []
ACTIVE_EVICTION = False
ORIGINAL_LLAMA_FORWARD = LlamaAttention.forward


def select_pyramid_kv(query_states, key_states, value_states, layer_idx, groups, scaling):
    cache_len = key_states.shape[-2]
    budget = min(PYRAMID_BUDGETS[layer_idx], cache_len)
    if cache_len <= budget:
        return key_states, value_states
    obs = min(WINDOW, cache_len)
    old_len = cache_len - obs
    old_keep = max(budget - obs, 0)
    recent_query = query_states[:, :, -obs:, :]
    repeated_key = repeat_kv(key_states, groups) if groups > 1 else key_states
    scores = torch.matmul(recent_query, repeated_key.transpose(2, 3)) * scaling
    causal = torch.triu(
        torch.full((obs, obs), float("-inf"), device=scores.device, dtype=scores.dtype),
        diagonal=1,
    )
    scores[:, :, :, -obs:] += causal[None, None, :, :]
    probabilities = F.softmax(scores, dim=-1, dtype=torch.float32)
    prefix_scores = probabilities[:, :, :, :old_len].sum(dim=-2)
    prefix_scores = prefix_scores.reshape(
        prefix_scores.shape[0], key_states.shape[1], groups, old_len,
    ).mean(dim=2)
    prefix_scores = F.avg_pool1d(
        prefix_scores, kernel_size=POOL_KERNEL, stride=1, padding=POOL_KERNEL // 2,
    )
    topk = prefix_scores.topk(old_keep, dim=-1).indices.sort(dim=-1).values
    gather_index = topk.unsqueeze(-1).expand(-1, -1, -1, key_states.shape[-1])
    selected_key = torch.cat(
        [key_states[:, :, :old_len, :].gather(2, gather_index), key_states[:, :, -obs:, :]],
        dim=2,
    ).contiguous()
    selected_value = torch.cat(
        [value_states[:, :, :old_len, :].gather(2, gather_index), value_states[:, :, -obs:, :]],
        dim=2,
    ).contiguous()
    PKV_LOG.append({
        "layer": int(layer_idx),
        "before": int(cache_len),
        "after": int(selected_key.shape[-2]),
        "budget": int(budget),
        "head_selection_differs": bool((topk[:, 1:] != topk[:, :-1]).any().item()),
    })
    return selected_key, selected_value


def cache_wrapper_forward(
    self,
    hidden_states,
    position_embeddings=None,
    attention_mask=None,
    past_key_values=None,
    **kwargs,
):
    input_shape = hidden_states.shape[:-1]
    hidden_shape = (*input_shape, -1, self.head_dim)
    query_states = self.q_proj(hidden_states).view(hidden_shape).transpose(1, 2)
    key_states = self.k_proj(hidden_states).view(hidden_shape).transpose(1, 2)
    value_states = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)
    cos, sin = position_embeddings
    query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)
    is_prefill = (
        past_key_values is not None
        and past_key_values.get_seq_length(self.layer_idx) == 0
    )
    if past_key_values is not None and not is_prefill:
        input_tokens = int(key_states.shape[-2])
        key_states, value_states = past_key_values.update(
            key_states, value_states, self.layer_idx,
        )
        CACHE_UPDATE_LOG.append({
            "phase": "decode",
            "layer": int(self.layer_idx),
            "input_tokens": input_tokens,
            "returned_tokens": int(key_states.shape[-2]),
            "cache_type": type(past_key_values).__name__,
        })
    attention_tokens = int(key_states.shape[-2])
    attention_interface = ALL_ATTENTION_FUNCTIONS.get_interface(
        self.config._attn_implementation, eager_attention_forward,
    )
    attention_output, attention_weights = attention_interface(
        self,
        query_states,
        key_states,
        value_states,
        attention_mask,
        dropout=0.0,
        scaling=self.scaling,
        **kwargs,
    )
    if is_prefill:
        stored_key, stored_value = key_states, value_states
        if ACTIVE_EVICTION:
            stored_key, stored_value = select_pyramid_kv(
                query_states,
                key_states,
                value_states,
                self.layer_idx,
                self.num_key_value_groups,
                self.scaling,
            )
        update_input_tokens = int(stored_key.shape[-2])
        returned_key, returned_value = past_key_values.update(
            stored_key, stored_value, self.layer_idx,
        )
        CACHE_UPDATE_LOG.append({
            "phase": "prefill",
            "layer": int(self.layer_idx),
            "attention_tokens": attention_tokens,
            "update_input_tokens": update_input_tokens,
            "returned_tokens": int(returned_key.shape[-2]),
            "cache_type": type(past_key_values).__name__,
        })
        del returned_key, returned_value
    attention_output = attention_output.reshape(*input_shape, -1).contiguous()
    return self.o_proj(attention_output), attention_weights


def enable_wrapper(use_eviction):
    global ACTIVE_EVICTION
    assert LlamaAttention.forward is ORIGINAL_LLAMA_FORWARD
    PKV_LOG.clear()
    CACHE_UPDATE_LOG.clear()
    ACTIVE_EVICTION = bool(use_eviction)
    LlamaAttention.forward = cache_wrapper_forward


def disable_wrapper():
    global ACTIVE_EVICTION
    LlamaAttention.forward = ORIGINAL_LLAMA_FORWARD
    ACTIVE_EVICTION = False
    PKV_LOG.clear()
    CACHE_UPDATE_LOG.clear()
    assert LlamaAttention.forward is ORIGINAL_LLAMA_FORWARD


def new_cache(variant):
    return TurboQuantCache(bits=DEFAULT_TQ_BITS) if VARIANT_PATHS[variant]["tq"] else DynamicCache()


def cache_lengths(cache):
    lengths = []
    for layer in cache.layers:
        if hasattr(layer, "_total_len"):
            lengths.append(int(layer._total_len))
        else:
            lengths.append(int(layer.get_seq_length()))
    return lengths


def named_cache_tensors(cache):
    for layer_index, layer in enumerate(cache.layers):
        for name, value in vars(layer).items():
            if isinstance(value, torch.Tensor):
                yield layer_index, name, value
            elif isinstance(value, Mapping):
                for nested_name, nested_value in value.items():
                    if isinstance(nested_value, torch.Tensor):
                        yield layer_index, f"{name}.{nested_name}", nested_value
            elif isinstance(value, (list, tuple)):
                for nested_index, nested_value in enumerate(value):
                    if isinstance(nested_value, torch.Tensor):
                        yield layer_index, f"{name}.{nested_index}", nested_value


def storage_bytes(tensors):
    seen = set()
    total = 0
    for tensor in tensors:
        if not isinstance(tensor, torch.Tensor) or tensor.numel() == 0:
            continue
        storage = tensor.untyped_storage()
        key = (str(tensor.device), int(storage.data_ptr()), int(storage.nbytes()))
        if key not in seen:
            seen.add(key)
            total += int(storage.nbytes())
    return total


def cache_storage_metrics(cache, prefix):
    named = list(named_cache_tensors(cache))
    all_tensors = [tensor for _, _, tensor in named]
    indices = [tensor for _, name, tensor in named if name in {"_key_indices", "_value_indices"}]
    norms = [tensor for _, name, tensor in named if name in {"_key_norms", "_value_norms"}]
    residuals = [tensor for _, name, tensor in named if name in {"_residual_keys", "_residual_values"}]
    materialized = [tensor for _, name, tensor in named if name in {"keys", "values"}]
    payload = indices + norms + residuals
    package_bytes = 0
    if callable(getattr(cache, "memory_usage_bytes", None)):
        package_bytes = int(cache.memory_usage_bytes().get("total_bytes", 0))
    return {
        f"{prefix}_cache_live_bytes": storage_bytes(all_tensors),
        f"{prefix}_tq_index_bytes": storage_bytes(indices),
        f"{prefix}_tq_norm_bytes": storage_bytes(norms),
        f"{prefix}_tq_residual_bytes": storage_bytes(residuals),
        f"{prefix}_tq_payload_bytes": storage_bytes(payload),
        f"{prefix}_tq_materialized_bytes": storage_bytes(materialized),
        f"{prefix}_tq_package_reported_bytes": package_bytes,
        f"{prefix}_tensor_count": len(all_tensors),
    }


TQ_STATE_NAMES = {
    "_key_indices", "_value_indices", "_key_norms", "_value_norms",
    "_residual_keys", "_residual_values",
}


def assert_no_tq_state(cache):
    assert isinstance(cache, DynamicCache) and not isinstance(cache, TurboQuantCache)
    for _, name, _ in named_cache_tensors(cache):
        assert name not in TQ_STATE_NAMES


def assert_no_compression_state(cache):
    assert ACTIVE_EVICTION is False
    assert not PKV_LOG
    assert_no_tq_state(cache)


def validate_cache_path(cache, variant, prefill_lengths, decode_tokens, update_log):
    path = VARIANT_PATHS[variant]
    assert len(cache.layers) == NUM_LAYERS
    expected_prefill = (
        [min(CONTEXT_TOKENS, budget) for budget in PYRAMID_BUDGETS]
        if path["eviction"] else [CONTEXT_TOKENS] * NUM_LAYERS
    )
    assert prefill_lengths == expected_prefill
    final_lengths = cache_lengths(cache)
    assert final_lengths == [length + decode_tokens for length in expected_prefill]
    prefill_updates = [row for row in update_log if row["phase"] == "prefill"]
    decode_updates = [row for row in update_log if row["phase"] == "decode"]
    if path["wrapper"]:
        assert len(prefill_updates) == NUM_LAYERS
        assert all(row["attention_tokens"] == CONTEXT_TOKENS for row in prefill_updates)
        assert [row["update_input_tokens"] for row in prefill_updates] == expected_prefill
        assert len(decode_updates) == NUM_LAYERS * decode_tokens
        assert all(row["input_tokens"] == 1 for row in decode_updates)
        assert sorted({row["layer"] for row in decode_updates}) == (
            list(range(NUM_LAYERS)) if decode_tokens else []
        )
    else:
        assert not update_log
    if path["eviction"]:
        assert len(PKV_LOG) == NUM_LAYERS
        assert sorted(row["layer"] for row in PKV_LOG) == list(range(NUM_LAYERS))
        assert all(row["before"] == CONTEXT_TOKENS for row in PKV_LOG)
        assert [row["after"] for row in sorted(PKV_LOG, key=lambda row: row["layer"])] == expected_prefill
    else:
        assert not PKV_LOG
    quantized_layers = 0
    if path["tq"]:
        assert isinstance(cache, TurboQuantCache) and cache.bits == DEFAULT_TQ_BITS
        for layer, total_length in zip(cache.layers, final_lengths):
            assert layer.bits == DEFAULT_TQ_BITS
            quantized_tokens = max(total_length - layer.residual_len, 0)
            residual_tokens = min(total_length, layer.residual_len)
            index_shape = (1, NUM_KV_HEADS, quantized_tokens, HEAD_DIM)
            norm_shape = (1, NUM_KV_HEADS, quantized_tokens, 1)
            if quantized_tokens:
                quantized_layers += 1
                assert tuple(layer._key_indices.shape) == index_shape
                assert tuple(layer._value_indices.shape) == index_shape
                assert tuple(layer._key_norms.shape) == norm_shape
                assert tuple(layer._value_norms.shape) == norm_shape
                for indices in (layer._key_indices, layer._value_indices):
                    assert indices.dtype == torch.uint8
                    assert int(indices.max()) < 2 ** DEFAULT_TQ_BITS
                for norms in (layer._key_norms, layer._value_norms):
                    assert torch.isfinite(norms).all() and (norms >= 0).all()
            else:
                assert layer._key_indices.numel() == layer._value_indices.numel() == 0
            assert layer._residual_keys.shape[-2] == residual_tokens
            assert layer._residual_values.shape[-2] == residual_tokens
            assert layer.keys.shape[-2] == total_length
            assert layer.values.shape[-2] == total_length
            assert torch.isfinite(layer.keys).all() and torch.isfinite(layer.values).all()
        assert quantized_layers == sum(length > 128 for length in final_lengths)
        metrics = cache_storage_metrics(cache, "validation")
        assert metrics["validation_tq_index_bytes"] > 0
        assert metrics["validation_tq_norm_bytes"] > 0
        assert metrics["validation_tq_residual_bytes"] > 0
        assert metrics["validation_tq_materialized_bytes"] > 0
    else:
        assert_no_tq_state(cache)
        if not path["eviction"]:
            assert_no_compression_state(cache)
    return {
        "structural_pass": True,
        "quantized_layers": quantized_layers,
        "evicted_layers": NUM_LAYERS if path["eviction"] else 0,
        "cache_min_prefill": min(prefill_lengths),
        "cache_max_prefill": max(prefill_lengths),
        "cache_min_after": min(final_lengths),
        "cache_max_after": max(final_lengths),
        "active_decode_cache_updates": len(decode_updates),
    }




def prompt_tensor(prompt, device):
    ids = torch.tensor([prompt["input_ids"]], dtype=torch.long, device=device)
    actual_hash = token_ids_hash(ids[0].detach().cpu().tolist())
    assert ids.shape == (1, CONTEXT_TOKENS)
    assert actual_hash == prompt["input_ids_sha256"]
    return ids, actual_hash


def run_greedy_path(model, variant, prompt, max_new_tokens=MAX_NEW_TOKENS):
    path = VARIANT_PATHS[variant]
    if path["wrapper"]:
        enable_wrapper(path["eviction"])
    else:
        assert LlamaAttention.forward is ORIGINAL_LLAMA_FORWARD
        PKV_LOG.clear()
        CACHE_UPDATE_LOG.clear()
    cache = new_cache(variant)
    ids, actual_hash = prompt_tensor(prompt, model.device)
    generated = []
    try:
        with torch.inference_mode():
            output = model(
                input_ids=ids, past_key_values=cache, use_cache=True, logits_to_keep=1,
            )
            prefill_lengths = cache_lengths(cache)
            prefill_logits = output.logits[0, -1].float().detach().cpu()
            next_token = output.logits[:, -1, :].argmax(-1, keepdim=True)
            for step in range(max_new_tokens):
                generated.append(next_token)
                if next_token.item() in np.atleast_1d(tokenizer.eos_token_id):
                    break
                if step == max_new_tokens - 1:
                    break
                absolute_position = CONTEXT_TOKENS + step
                position = torch.tensor([[absolute_position]], device=model.device)
                cache_position = torch.tensor([absolute_position], device=model.device)
                output = model(
                    input_ids=next_token,
                    position_ids=position,
                    cache_position=cache_position,
                    past_key_values=cache,
                    use_cache=True,
                    logits_to_keep=1,
                )
                next_token = output.logits[:, -1, :].argmax(-1, keepdim=True)
        output_ids = torch.cat(generated, dim=1)[0].tolist()
        update_log = list(CACHE_UPDATE_LOG)
        structure = validate_cache_path(
            cache, variant, prefill_lengths, max(0, len(output_ids) - 1), update_log,
        )
        storage = cache_storage_metrics(cache, "generation_final")
        return {
            "input_ids_sha256": actual_hash,
            "prefill_logits": prefill_logits,
            "prefill_top1": int(prefill_logits.argmax()),
            "response": tokenizer.decode(output_ids, skip_special_tokens=True),
            "output_token_ids": output_ids,
            "prefill_lengths": prefill_lengths,
            "final_lengths": cache_lengths(cache),
            "structure": structure,
            "storage": storage,
        }
    finally:
        if path["wrapper"]:
            disable_wrapper()
        del cache, ids
        clear_gpu()


def verify_baseline_parity(model):
    stock = run_greedy_path(model, "C1", PROMPT)
    wrapped = run_greedy_path(model, "C1-W", PROMPT)
    max_difference = float((stock["prefill_logits"] - wrapped["prefill_logits"]).abs().max())
    same_logits = torch.allclose(
        stock["prefill_logits"], wrapped["prefill_logits"], atol=PARITY_ATOL, rtol=PARITY_RTOL
    )
    same_tokens = stock["output_token_ids"] == wrapped["output_token_ids"]
    same_cache = (
        stock["prefill_lengths"] == wrapped["prefill_lengths"]
        and stock["final_lengths"] == wrapped["final_lengths"]
    )
    assert same_logits and same_tokens and same_cache, "Stock/wrapped parity probe failed"
    return {
        "passed": True,
        "input_ids_sha256": EXPECTED_PROMPT_SHA256,
        "max_abs_prefill_logit_difference": max_difference,
        "atol": PARITY_ATOL,
        "rtol": PARITY_RTOL,
        "greedy_token_ids_equal": same_tokens,
        "full_cache_lengths_equal": same_cache,
    }

## 4. Define the power measurement

Prefill and decode are measured separately, as in the existing study. Their raw integrated joules are added. Each measurement keeps its trace, block order, cache validation and workload hashes. A block is saved only after all five variants pass.

In [5]:
def nvml_value(function, *args, default=np.nan):
    try:
        return function(*args)
    except (AttributeError, pynvml.NVMLError):
        return default


def sensor_sample():
    memory = pynvml.nvmlDeviceGetMemoryInfo(NVML_HANDLE)
    return {
        "power_w": float(pynvml.nvmlDeviceGetPowerUsage(NVML_HANDLE) / 1000.0),
        "nvml_used_mb": float(memory.used / 2**20),
        "temperature_c": float(nvml_value(
            pynvml.nvmlDeviceGetTemperature,
            NVML_HANDLE,
            pynvml.NVML_TEMPERATURE_GPU,
        )),
    }


def telemetry_snapshot():
    sample = sensor_sample()
    sample.update({
        "sm_clock_mhz": float(nvml_value(
            pynvml.nvmlDeviceGetClockInfo, NVML_HANDLE, pynvml.NVML_CLOCK_SM,
        )),
        "memory_clock_mhz": float(nvml_value(
            pynvml.nvmlDeviceGetClockInfo, NVML_HANDLE, pynvml.NVML_CLOCK_MEM,
        )),
        "power_limit_w": float(nvml_value(
            pynvml.nvmlDeviceGetEnforcedPowerLimit, NVML_HANDLE,
        )) / 1000.0,
        "throttle_reasons": int(nvml_value(
            pynvml.nvmlDeviceGetCurrentClocksThrottleReasons,
            NVML_HANDLE,
            default=0,
        )),
    })
    return sample


class GPUMonitor:
    def __init__(self, interval_s=0.1):
        self.interval_s = float(interval_s)
        self.samples = []
        self.running = False
        self.stop_event = threading.Event()
        self.lock = threading.Lock()

    def sample(self, elapsed_s=None):
        row = sensor_sample()
        row["elapsed_s"] = float(
            time.perf_counter() - self.t0 if elapsed_s is None else elapsed_s
        )
        with self.lock:
            self.samples.append(row)

    def poll(self):
        deadline = self.t0 + self.interval_s
        while not self.stop_event.wait(max(0.0, deadline - time.perf_counter())):
            try:
                self.sample()
            except pynvml.NVMLError:
                pass
            deadline += self.interval_s

    def start(self):
        assert not self.running
        torch.cuda.synchronize()
        self.samples = []
        self.stop_event.clear()
        self.t0 = time.perf_counter()
        self.sample(elapsed_s=0.0)
        self.running = True
        self.thread = threading.Thread(target=self.poll, daemon=True)
        self.thread.start()

    def stop(self):
        assert self.running
        torch.cuda.synchronize()
        self.stop_event.set()
        self.thread.join(timeout=2)
        if self.thread.is_alive():
            raise RuntimeError("Power monitor thread did not terminate")
        self.sample()
        self.running = False
        with self.lock:
            samples = sorted(self.samples, key=lambda row: row["elapsed_s"])
        assert len(samples) >= 2
        times = np.asarray([row["elapsed_s"] for row in samples], dtype=float)
        powers = np.asarray([row["power_w"] for row in samples], dtype=float)
        assert np.all(np.diff(times) >= 0) and np.isfinite(powers).all()
        duration = float(times[-1] - times[0])
        energy = float(np.trapezoid(powers, times))
        temperatures = [
            row["temperature_c"] for row in samples if np.isfinite(row["temperature_c"])
        ]
        return {
            "energy_j": energy,
            "duration_s": duration,
            "avg_power_w": energy / duration,
            "max_power_w": float(powers.max()),
            "nvml_peak_mb": float(max(row["nvml_used_mb"] for row in samples)),
            "max_temperature_c": float(max(temperatures)) if temperatures else np.nan,
            "n_samples": len(samples),
            "samples": samples,
        }


def measure_run(model, variant, block, position, attempt, warmup=False):
    path = VARIANT_PATHS[variant]
    assert LlamaAttention.forward is ORIGINAL_LLAMA_FORWARD
    clear_gpu()
    before = telemetry_snapshot()
    assert before["power_limit_w"] == GPU_IDENTITY["power_limit_w"]
    ids, prompt_hash = prompt_tensor(PROMPT, model.device)
    cache = new_cache(variant)
    monitors = []
    if path["wrapper"]:
        enable_wrapper(path["eviction"])
    try:
        torch.cuda.reset_peak_memory_stats()
        prefill_monitor = GPUMonitor(SAMPLE_INTERVAL_S)
        monitors.append(prefill_monitor)
        prefill_monitor.start()
        prefill_start = time.perf_counter()
        with torch.inference_mode():
            model(input_ids=ids, past_key_values=cache, use_cache=True, logits_to_keep=1)
        torch.cuda.synchronize()
        prefill_s = time.perf_counter() - prefill_start
        prefill = prefill_monitor.stop()
        prefill_lengths = cache_lengths(cache)
        prefill_peak_allocated = torch.cuda.max_memory_allocated()
        torch.cuda.reset_peak_memory_stats()

        decode_monitor = GPUMonitor(SAMPLE_INTERVAL_S)
        monitors.append(decode_monitor)
        decode_monitor.start()
        decode_start = time.perf_counter()
        with torch.inference_mode():
            for step, token_id in enumerate(TRACE_IDS):
                absolute_position = CONTEXT_TOKENS + step
                token = torch.tensor([[token_id]], dtype=torch.long, device=model.device)
                position_ids = torch.tensor([[absolute_position]], device=model.device)
                cache_position = torch.tensor([absolute_position], device=model.device)
                model(
                    input_ids=token, position_ids=position_ids, cache_position=cache_position,
                    past_key_values=cache, use_cache=True, logits_to_keep=1,
                )
        torch.cuda.synchronize()
        decode_s = time.perf_counter() - decode_start
        decode = decode_monitor.stop()
        decode_peak_allocated = torch.cuda.max_memory_allocated()
        structure = validate_cache_path(
            cache, variant, prefill_lengths, FIXED_DECODE_TOKENS, list(CACHE_UPDATE_LOG),
        )
        after = telemetry_snapshot()
        assert after["power_limit_w"] == GPU_IDENTITY["power_limit_w"]
        assert prefill["energy_j"] > 0 and decode["energy_j"] > 0

        row = {
            "fingerprint": EXPERIMENT_FINGERPRINT,
            "session_id": SESSION_ID,
            "block": block,
            "position": position,
            "variant": variant,
            "input_ids_sha256": prompt_hash,
            "trace_ids_sha256": TRACE_IDS_SHA256,
            "fixed_decode_tokens": FIXED_DECODE_TOKENS,
            "prefill_s": prefill_s,
            "decode_s": decode_s,
            "prefill_energy_j": prefill["energy_j"],
            "decode_energy_j": decode["energy_j"],
            "total_energy_j": prefill["energy_j"] + decode["energy_j"],
            "prefill_peak_allocated_bytes": prefill_peak_allocated,
            "decode_peak_allocated_bytes": decode_peak_allocated,
            "temperature_before_c": before["temperature_c"],
            "temperature_after_c": after["temperature_c"],
            "power_limit_w": after["power_limit_w"],
            **structure,
            **cache_storage_metrics(cache, "post_decode"),
        }
        if not warmup:
            trace = POWER_TRACE_DIR / f"block-{block:02d}-{position}-{variant}-{attempt}.json"
            atomic_write_json(trace, {
                "fingerprint": EXPERIMENT_FINGERPRINT,
                "session_id": SESSION_ID,
                "block": block,
                "position": position,
                "variant": variant,
                "input_ids_sha256": prompt_hash,
                "trace_ids_sha256": TRACE_IDS_SHA256,
                "prefill": prefill["samples"],
                "decode": decode["samples"],
            })
            row["trace"] = trace.relative_to(SAVE_DIR).as_posix()
            row["trace_sha256"] = sha256_file(trace)
        return row
    finally:
        try:
            for monitor in monitors:
                if monitor.running:
                    monitor.stop()
        finally:
            disable_wrapper()
            del cache, ids
            clear_gpu()

## 5. Run the experiment

Within every block, all five variants use one resident model and the same workload. The order balances treatment positions and consecutive pairs over ten blocks. One warm-up per variant checks its cache path and is excluded from analysis. If interrupted in the same kernel, rerun this cell to resume at the first incomplete block. A fresh kernel creates a fresh measurement session.

In [6]:
require_frozen_gate()
assert LlamaAttention.forward is ORIGINAL_LLAMA_FORWARD
model = load_awq()
try:
    parity = verify_baseline_parity(model)
    atomic_write_json(SAVE_DIR / "baseline_parity.json", parity)
    print("Stock and wrapped AWQ parity passed on the measured prompt.")

    for position, variant in enumerate(SCHEDULE[0]):
        measure_run(model, variant, block=-1, position=position, attempt="warmup", warmup=True)
    print("All five cache paths passed warm-up validation.")

    for block, order in enumerate(SCHEDULE, start=1):
        block_file = BLOCK_DIR / f"block-{block:02d}.json"
        if block_file.exists():
            print(f"Block {block}/{MEASURED_BLOCKS} already complete")
            continue
        require_frozen_gate()
        attempt = uuid.uuid4().hex[:8]
        rows = []
        for position, variant in enumerate(order):
            row = measure_run(model, variant, block, position, attempt)
            rows.append(row)
            print(f"Block {block}/{MEASURED_BLOCKS} | {variant} | {row['total_energy_j']:.2f} J")
        atomic_write_json(block_file, rows)
finally:
    disable_wrapper()
    del model
    clear_gpu()

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.25GB            

model.safetensors: downloading bytes:           |  0.00B            


INFO  LogBar: headless/CI mode; progress animations disabled.                   
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.
INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.
INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.           
INFO    ┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐ │ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │ └─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘ GPT-QModel   : 7.1.0 Transformers : 5.12.1 Torch        : 2.14.1+cu130 Triton       : 3.8.0
INFO  ExLlamaV2 AWQ: compiling torch.ops JIT extension in `/root/.cache/gptqmodel/torch_

Loading weights:   0%|          | 0/646 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

INFO  gc.collect() reclaimed 5746 objects in 0.372s                             


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Stock and wrapped AWQ parity passed on the measured prompt.
All five cache paths passed warm-up validation.
Block 1/10 | C1-W | 530.13 J
Block 1/10 | C3-Ref | 840.99 J
Block 1/10 | C4-Ref | 596.50 J
Block 1/10 | C2 | 558.79 J
Block 1/10 | C1 | 579.33 J
Block 2/10 | C3-Ref | 880.89 J
Block 2/10 | C2 | 565.16 J
Block 2/10 | C1-W | 583.83 J
Block 2/10 | C1 | 578.50 J
Block 2/10 | C4-Ref | 653.76 J
Block 3/10 | C1-W | 575.43 J
Block 3/10 | C4-Ref | 621.52 J
Block 3/10 | C3-Ref | 883.15 J
Block 3/10 | C1 | 572.82 J
Block 3/10 | C2 | 566.70 J
Block 4/10 | C3-Ref | 880.51 J
Block 4/10 | C1-W | 576.75 J
Block 4/10 | C2 | 541.63 J
Block 4/10 | C4-Ref | 625.44 J
Block 4/10 | C1 | 587.24 J
Block 5/10 | C1 | 585.19 J
Block 5/10 | C4-Ref | 656.32 J
Block 5/10 | C2 | 540.73 J
Block 5/10 | C1-W | 589.88 J
Block 5/10 | C3-Ref | 880.17 J
Block 6/10 | C2 | 552.37 J
Block 6/10 | C1 | 582.89 J
Block 6/10 | C3-Ref | 880.67 J
Block 6/10 | C4-Ref | 638.68 J
Block 6/10 | C1-W | 574.44 J
Block 7/10 | C4-Ref | 

## 6. Verify traces and calculate the paired estimates

Report C1-W minus C1 and both interaction estimates with 95% paired t intervals. The mean and interval are calculated from the ten complete blocks. The sign of each point estimate and whether its interval crosses zero are reported separately.

In [7]:
def integrate_trace(samples):
    assert len(samples) >= 2 and samples[0]["elapsed_s"] == 0.0
    times = [sample["elapsed_s"] for sample in samples]
    powers = [sample["power_w"] for sample in samples]
    assert all(math.isfinite(t) for t in times)
    assert all(math.isfinite(p) and p >= 0 for p in powers)
    assert all(a <= b for a, b in zip(times, times[1:])) and times[-1] > 0
    return math.fsum(
        (b - a) * (x + y) / 2 for a, b, x, y in zip(times, times[1:], powers, powers[1:])
    )


def read_completed_rows():
    output = []
    used_traces = set()
    for block, order in enumerate(SCHEDULE, start=1):
        path = BLOCK_DIR / f"block-{block:02d}.json"
        if not path.is_file():
            raise RuntimeError(f"Block {block} is incomplete. Finish the measurement cell first.")
        rows = json.loads(path.read_text())
        assert [row["variant"] for row in rows] == order
        for position, row in enumerate(rows):
            assert row["block"] == block and row["position"] == position
            assert row["fingerprint"] == EXPERIMENT_FINGERPRINT and row["session_id"] == SESSION_ID
            assert row["input_ids_sha256"] == EXPECTED_PROMPT_SHA256
            assert row["trace_ids_sha256"] == EXPECTED_TRACE_IDS_SHA256
            assert row["fixed_decode_tokens"] == FIXED_DECODE_TOKENS and row["structural_pass"]
            assert row["power_limit_w"] == GPU_IDENTITY["power_limit_w"]
            trace_path = (SAVE_DIR / row["trace"]).resolve()
            assert trace_path.parent == POWER_TRACE_DIR.resolve()
            assert trace_path not in used_traces and sha256_file(trace_path) == row["trace_sha256"]
            trace = json.loads(trace_path.read_text())
            for key in ("fingerprint", "session_id", "block", "position", "variant", "input_ids_sha256", "trace_ids_sha256"):
                assert trace[key] == row[key]
            phases = [integrate_trace(trace[phase]) for phase in ("prefill", "decode")]
            for phase, value in zip(("prefill", "decode"), phases):
                assert math.isclose(value, row[f"{phase}_energy_j"], rel_tol=1e-10, abs_tol=1e-8)
            assert math.isclose(sum(phases), row["total_energy_j"], rel_tol=1e-10, abs_tol=1e-8)
            used_traces.add(trace_path)
            output.append(row)
    assert len(output) == len(used_traces) == MEASURED_BLOCKS * len(VARIANTS)
    return output


def block_contrasts(rows):
    paired = []
    for block in range(1, MEASURED_BLOCKS + 1):
        energy = {row["variant"]: row["total_energy_j"] for row in rows if row["block"] == block}
        assert set(energy) == set(VARIANTS)
        eviction_with_tq = energy["C4-Ref"] - energy["C3-Ref"]
        stock_i = eviction_with_tq - (energy["C2"] - energy["C1"])
        wrapped_i = eviction_with_tq - (energy["C2"] - energy["C1-W"])
        delta = energy["C1-W"] - energy["C1"]
        assert math.isclose(wrapped_i - stock_i, delta, rel_tol=1e-10, abs_tol=1e-8)
        paired.append({
            "block": block,
            "wrapper_delta_j": delta,
            "stock_interaction_j": stock_i,
            "wrapped_interaction_j": wrapped_i,
        })
    return paired


def summarize(values):
    n = len(values)
    assert n >= 2 and all(math.isfinite(value) for value in values)
    mean = float(np.mean(values))
    std = float(np.std(values, ddof=1))
    half_width = float(stats.t.ppf(0.975, n - 1)) * std / math.sqrt(n)
    return {"blocks": n, "mean_j": mean, "sd_j": std, "ci95_low_j": mean - half_width, "ci95_high_j": mean + half_width}


def write_csv(path, rows):
    temporary = Path(path).with_suffix(".csv.tmp")
    with temporary.open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(path)


def results_note(summaries, by_variant):
    delta = summaries["wrapper_delta_j"]
    shift_pct = 100 * delta["mean_j"] / by_variant["C1"]["mean_j"]

    def interval(estimate):
        mean = estimate["mean_j"]
        low = estimate["ci95_low_j"]
        high = estimate["ci95_high_j"]
        includes_zero = "includes zero" if low <= 0 <= high else "excludes zero"
        return f"{mean:+.2f} J (95% paired CI [{low:+.2f}, {high:+.2f}] J, {includes_zero})"

    stock = summaries["stock_interaction_j"]
    wrapped = summaries["wrapped_interaction_j"]
    both_negative = stock["mean_j"] < 0 and wrapped["mean_j"] < 0
    if both_negative:
        interpretation = "The interaction estimate stayed negative with either baseline."
    else:
        interpretation = "Use the signed estimates and intervals above to assess the change with baseline choice."

    return (
        "# Same-session Llama wrapper control and energy interaction\n\n"
        f"We measured all five conditions in {MEASURED_BLOCKS} counterbalanced blocks on one NVIDIA T4 "
        f"at an enforced board power limit of {GPU_IDENTITY['power_limit_w']:g} W, "
        "using the same resident AWQ checkpoint, 4,096-token prompt, and 64 teacher-forced decode tokens. "
        "Both AWQ energy baselines had cache compression disabled. The stock/wrapped parity probe passed. "
        "Energy is raw GPU-board joules integrated over prefill and decode.\n\n"
        f"- Stock AWQ C1 mean ± SD: {by_variant['C1']['mean_j']:.2f} ± {by_variant['C1']['sd_j']:.2f} J.\n"
        f"- Wrapped AWQ C1-W mean ± SD: {by_variant['C1-W']['mean_j']:.2f} ± {by_variant['C1-W']['sd_j']:.2f} J.\n"
        f"- Wrapped minus stock baseline is {interval(delta)}, or {shift_pct:+.2f}% of the stock mean.\n"
        f"- Interaction using stock C1 is {interval(stock)}.\n"
        f"- Interaction using wrapped C1-W is {interval(wrapped)}.\n\n"
        f"Substituting the wrapped baseline shifts the mean interaction by {delta['mean_j']:+.2f} J. "
        f"{interpretation} These intervals describe repeated blocks on the tested workload and session. "
        "The experiment quantifies the energy effect of the tested wrapper configuration. "
        "Published PyramidKV accuracy is outside this test.\n"
    )


parity_path = SAVE_DIR / "baseline_parity.json"
assert json.loads(parity_path.read_text())["passed"]
rows = read_completed_rows()
paired = block_contrasts(rows)
metrics = ("wrapper_delta_j", "stock_interaction_j", "wrapped_interaction_j")
summaries = {metric: summarize([r[metric] for r in paired]) for metric in metrics}
by_variant = {
    variant: summarize([r["total_energy_j"] for r in rows if r["variant"] == variant])
    for variant in VARIANTS
}
write_csv(SAVE_DIR / "matched_energy_rows.csv", rows)
write_csv(SAVE_DIR / "paired_block_contrasts.csv", paired)
atomic_write_json(SAVE_DIR / "result_summary.json", {
    "fingerprint": EXPERIMENT_FINGERPRINT, "by_variant": by_variant, "paired_contrasts": summaries,
})
note = results_note(summaries, by_variant)
(SAVE_DIR / "reviewer_summary.md").write_text(note)
bundle_paths = [
    SAVE_DIR / name for name in (
        "run_manifest.json", "workload.json", "baseline_parity.json", "matched_energy_rows.csv",
        "paired_block_contrasts.csv", "result_summary.json", "reviewer_summary.md",
    )
] + sorted(BLOCK_DIR.glob("block-*.json")) + [SAVE_DIR / row["trace"] for row in rows]
atomic_write_json(SAVE_DIR / "final_integrity.json", {
    "passed": True, "fingerprint": EXPERIMENT_FINGERPRINT, "session_id": SESSION_ID,
    "complete_blocks": MEASURED_BLOCKS, "measurement_rows": len(rows), "verified_traces": len(rows),
    "file_sha256": {path.relative_to(SAVE_DIR).as_posix(): sha256_file(path) for path in bundle_paths},
})
print(note)

# Same-session Llama wrapper control and energy interaction

We measured all five conditions in 10 counterbalanced blocks on one NVIDIA T4 at an enforced board power limit of 70 W, using the same resident AWQ checkpoint, 4,096-token prompt, and 64 teacher-forced decode tokens. Both AWQ energy baselines had cache compression disabled. The stock/wrapped parity probe passed. Energy is raw GPU-board joules integrated over prefill and decode.

- Stock AWQ C1 mean ± SD: 579.26 ± 5.03 J.
- Wrapped AWQ C1-W mean ± SD: 575.13 ± 16.71 J.
- Wrapped minus stock baseline is -4.13 J (95% paired CI [-16.36, +8.10] J, includes zero), or -0.71% of the stock mean.
- Interaction using stock C1 is -212.41 J (95% paired CI [-226.65, -198.16] J, excludes zero).
- Interaction using wrapped C1-W is -216.54 J (95% paired CI [-237.15, -195.92] J, excludes zero).

Substituting the wrapped baseline shifts the mean interaction by -4.13 J. The interaction estimate stayed negative with either baseline. These interva

## 7. Download the verified result bundle

The ZIP contains the complete blocks, referenced traces, and their hashes. Incomplete attempts are excluded. The generated Markdown summary is written from the observed results.

In [8]:
from zipfile import ZIP_DEFLATED, ZipFile

integrity = json.loads((SAVE_DIR / "final_integrity.json").read_text())
assert integrity["passed"] and integrity["fingerprint"] == EXPERIMENT_FINGERPRINT
zip_path = SAVE_DIR.with_suffix(".zip")
with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as archive:
    archive.write(SAVE_DIR / "final_integrity.json", "final_integrity.json")
    for name, digest in integrity["file_sha256"].items():
        path = SAVE_DIR / name
        assert sha256_file(path) == digest
        archive.write(path, name)
with ZipFile(zip_path) as archive:
    assert archive.testzip() is None
print(f"Verified result bundle: {zip_path}")
from google.colab import files
files.download(str(zip_path))

Verified result bundle: /content/CSE407_llama_matched_wrapper_interaction_v1_20261122T112205Z-348460e2.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>